In [33]:
import slippi as slp
import glob
import os
import shutil

folder = 'bonk'
folder = 'swift'
folder = 'players'

slp_base_path = f"/home/agiera/Slippi/{folder}/"
video_base_path = f"/home/agiera/ssbm/vods/{folder}/"

tmp_path = "/tmp/slp-event-search/"

In [34]:
player = 'electroman'
player = "byrn"
player = 'motobug'
player = 'oz'
player = 'regex'
player = 'cruor'
player = 'bbats'
player = 'ginger'
player = 'blesse'
# use this spreadsheet to find action states
# https://docs.google.com/spreadsheets/d/1JX2w-r2fuvWuNgGb6D3Cs4wHQKLFegZe2jhbBuIhCG8/preview#gid=13
state_filter = [slp.id.ActionState.ATTACK_LW_4]  # downsmash
state_filter = [slp.id.ActionState.CLIFF_CATCH]
state_filter = [
    # slp.id.ActionState.GUARD_SET_OFF,
    slp.id.ActionState.DAMAGE_FALL,
    slp.id.ActionState.FALL_SPECIAL, slp.id.ActionState.FALL_SPECIAL_F, slp.id.ActionState.FALL_SPECIAL_B,
    slp.id.ActionState.FALL, slp.id.ActionState.FALL_F, slp.id.ActionState.FALL_B
]
character_filter = slp.id.CSSCharacter.PIKACHU
match_character_filter = [
    slp.id.CSSCharacter.GANONDORF,
    slp.id.CSSCharacter.BOWSER,
    character_filter,
]
interval = (0, 300)
filter_off_stage = True

In [35]:
distance_from_stage_edge = 10
stage_radii = {
    slp.id.Stage.FINAL_DESTINATION: 85.5606,
    slp.id.Stage.BATTLEFIELD: 68.4000,
    slp.id.Stage.YOSHIS_STORY: 56.0000,
    slp.id.Stage.FOUNTAIN_OF_DREAMS: 63.3500,
    slp.id.Stage.POKEMON_STADIUM: 87.7500,
    slp.id.Stage.DREAM_LAND_N64: 77.2700,
}

In [36]:
search_pattern = f"{slp_base_path}*{player}/*.slp"
output_filepath = f"{video_base_path}/{player}-{character_filter.name}-{state_filter[0].name}.mp4"

shutil.rmtree(tmp_path, ignore_errors=True)
os.makedirs(tmp_path, exist_ok=True)

In [ ]:
def character_ports(game: slp.Game, character: slp.id.CSSCharacter):
    ports = []
    for i, player in enumerate(game.start.players):
        if player is None:
            continue
        if player.character == character:
            ports.append(i)
    return ports

In [38]:
events = {}
for filepath in glob.iglob(search_pattern):
    events[filepath] = []
    try:
        game = slp.Game(filepath)
    except Exception as e:
        print(f"Invalid Slippi file: {filepath} - Error: {e}")
        continue
    if not all([
        player.character in match_character_filter
        for player in game.start.players
        if player is not None
    ]):
        print(f"Incorrect matchup: {filepath}")
        continue
    print(f"Found valid Slippi file: {filepath}")

    character_ports_list = character_ports(game, character_filter)
    if not character_ports_list:
        print(f"No {character_filter.name} found in file: {filepath}")
        continue
    for i in character_ports_list:
        for frame in game.frames:
            port = frame.ports[i]
            if port is None or port.leader is None:
                continue

            state = port.leader.pre.state

            if state not in state_filter:
                continue

            if filter_off_stage:
                stage_radius = stage_radii.get(game.start.stage, None)
                if stage_radius is None:
                    continue
                position = port.leader.post.position
                if abs(position.x) < stage_radius + distance_from_stage_edge:
                    continue

            # only take last frame filters hold
            if events[filepath] and events[filepath][-1] == frame.index - 1:
                events[filepath][-1] = frame.index
            else:
                print(f"Found event in file: {filepath} at frame {frame.index}")
                events[filepath].append(frame.index)

Found valid Slippi file: /home/agiera/Slippi/players/blesse/Game_20260201T015326.slp
Found event in file: /home/agiera/Slippi/players/blesse/Game_20260201T015326.slp at frame 573
Found event in file: /home/agiera/Slippi/players/blesse/Game_20260201T015326.slp at frame 688
Found event in file: /home/agiera/Slippi/players/blesse/Game_20260201T015326.slp at frame 723
Found event in file: /home/agiera/Slippi/players/blesse/Game_20260201T015326.slp at frame 789
Found event in file: /home/agiera/Slippi/players/blesse/Game_20260201T015326.slp at frame 1294
Found event in file: /home/agiera/Slippi/players/blesse/Game_20260201T015326.slp at frame 1323
Found event in file: /home/agiera/Slippi/players/blesse/Game_20260201T015326.slp at frame 3175
Found event in file: /home/agiera/Slippi/players/blesse/Game_20260201T015326.slp at frame 3309
Found event in file: /home/agiera/Slippi/players/blesse/Game_20260201T015326.slp at frame 8042
Found event in file: /home/agiera/Slippi/players/blesse/Game_202

In [39]:
clip_intervals = {}
for filepath, frames in events.items():
    clip_intervals[filepath] = []
    for frame in frames:
        start_frame = max(0, frame + interval[0])
        end_frame = frame + interval[1]
        # Avoid overlapping intervals
        if clip_intervals[filepath] and start_frame <= clip_intervals[filepath][-1][1]:
            clip_intervals[filepath][-1] = (clip_intervals[filepath][-1][0], end_frame)
        else:
            clip_intervals[filepath].append((start_frame, end_frame))

In [40]:
clip_filepaths = []
for filepath, intervals in clip_intervals.items():
    video_filepath = (
        filepath
        .replace(slp_base_path, video_base_path)
        .replace(".slp", ".mp4")
    )
    for start_frame, end_frame in intervals:
        # call ffmpeg to extract clip from video_filepath using start_frame and end_frame
        clip_filepath = f"{tmp_path}/clip-{os.path.basename(filepath)}-{start_frame}-{end_frame}.mp4"
        os.spawnvp(os.P_WAIT, "ffmpeg", [
            "ffmpeg",
            "-i", video_filepath,
            "-ss", str(start_frame / 60.0),
            "-to", str(end_frame / 60.0),
            "-c", "copy",
            "-y",
            clip_filepath,
        ])
        clip_filepaths.append(clip_filepath)
# Create a filelist for ffmpeg concatenation
with open(f"{tmp_path}/filelist.txt", "w") as f:
    for clip_filepath in clip_filepaths:
        f.write(f"file '{clip_filepath}'\n")
os.spawnvp(os.P_WAIT, "ffmpeg", [
    "ffmpeg",
    "-f", "concat",
    "-safe", "0",
    "-i", f"{tmp_path}/filelist.txt",
    "-c", "copy",
    "-y",
    output_filepath,
])

ffmpeg version 7.1.2 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15 (GCC)
  configuration: --prefix=/usr --bindir=/usr/bin --datadir=/usr/share/ffmpeg --docdir=/usr/share/doc/ffmpeg --incdir=/usr/include/ffmpeg --libdir=/usr/lib64 --mandir=/usr/share/man --arch=x86_64 --optflags='-O2 -flto=auto -ffat-lto-objects -fexceptions -g -grecord-gcc-switches -pipe -Wall -Wno-complain-wrong-lang -Werror=format-security -Wp,-U_FORTIFY_SOURCE,-D_FORTIFY_SOURCE=3 -Wp,-D_GLIBCXX_ASSERTIONS -specs=/usr/lib/rpm/redhat/redhat-hardened-cc1 -fstack-protector-strong -specs=/usr/lib/rpm/redhat/redhat-annobin-cc1 -m64 -march=x86-64 -mtune=generic -fasynchronous-unwind-tables -fstack-clash-protection -fcf-protection -mtls-dialect=gnu2 -fno-omit-frame-pointer -mno-omit-leaf-frame-pointer' --extra-ldflags='-Wl,-z,relro -Wl,--as-needed -Wl,-z,pack-relative-relocs -Wl,-z,now -specs=/usr/lib/rpm/redhat/redhat-hardened-ld -specs=/usr/lib/rpm/redhat/redhat-annobin-cc1 -Wl,--build-id=sha1 -specs=/

0